# EV Charging Cost Optimization (0701 Reproducible Edition)

## Research Notebook for 2026_Gu_EV_forecast

**Objective**: Compare forecasting methods under one fixed MPC core for a single benchmark day (2023-07-01), and reproduce publication figures/tables in one output folder.

**What this notebook guarantees**:
- One-click sequential execution from top to bottom
- Single-day benchmark (no mixed 3-day logic)
- All outputs saved under `2026_Gu_EV_forecast/`
- Unified plotting style (font sizes, colors, legends) for all key figures

**Data source priority**:
1. `2026_Gu_EV_forecast/clean_charging_sessions_enhanced.csv` (if exists)
2. `clean_charging_sessions.csv` (fallback)

**Benchmark day**: 2023-07-01

**Optimization horizon**: 24 hours (96 × 15-minute steps)

## Part 1: Imports, Style, and Global Controls

In [1]:
# Core imports (kept at the beginning to avoid undefined-symbol warnings later)
import os
import warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from pulp import *

warnings.filterwarnings("ignore")
%matplotlib inline

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Unified figure style (IEEE-friendly, color-blind-safe)
IEEE = {
    "blue": "#0072B2",
    "orange": "#D55E00",
    "green": "#009E73",
    "purple": "#CC79A7",
    "sky": "#56B4E9",
    "yellow": "#E69F00",
    "gray": "#4D4D4D",
    "black": "#000000",
}

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["legend.fontsize"] = 10

print("All imports loaded")
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("TensorFlow:", tf.__version__)

All imports loaded
NumPy: 2.4.4
Pandas: 3.0.2
TensorFlow: 2.21.0


In [2]:
# Global configuration (single-day, new output root)
workspace_root = "/Users/admin/Desktop/EV_program/2024Summer_EVResearch"
output_root = os.path.join(workspace_root, "2026_Gu_EV_forecast")

enhanced_path = os.path.join(output_root, "clean_charging_sessions_enhanced.csv")
clean_fallback = os.path.join(workspace_root, "clean_charging_sessions.csv")

CONFIG = {
    "workspace_root": workspace_root,
    "output_root": output_root,
    "fig_dir": os.path.join(output_root, "figures"),
    "table_dir": os.path.join(output_root, "tables"),
    "model_dir": os.path.join(output_root, "models"),
    "data_path": enhanced_path if os.path.exists(enhanced_path) else clean_fallback,

    # MPC horizon
    "T": 24,
    "N": 96,
    "delta_t": 24 / 96,
    "P_max": 6.6,

    # TOU window (16:00-21:00)
    "N_start_idx": int(16 / (24 / 96)) + 1,
    "N_end_idx": int(21 / (24 / 96)),

    # Tariff coefficients
    "r_energy_summer_onpeak": 0.11957 + 0.00671,
    "r_energy_summer_offpeak": 0.10008 + 0.00671,
    "r_power_summer_onpeak": 9.78 + 19.14,
    "r_power_nc": 24.48,
    "other_rate_1": 0.0578,
    "other_rate_2": 0.0058 + 0.00058 + 0.0003,

    # Single benchmark day
    "test_days": [pd.Timestamp("2023-07-01").date()],

    # Display method list (includes latest methods used in final comparison tables)
    "methods": [
        "Perfect", "Noforecast", "Persistence", "Statistic", "GMM",
        "LSTM", "TCN", "Transformer", "iTransformer",
        "iT-CP90", "iT-CP95", "TFT-Quantile", "DeepAR-Gaussian", "ITCN-ChargerAware",
    ],

    # Forecast module training defaults (for legacy deterministic bank)
    "lookback_days": 14,
    "dl_epochs": 15,
    "dl_batch_size": 16,
    "max_daily_sessions_for_solve": 160,
}

for p in [CONFIG["output_root"], CONFIG["fig_dir"], CONFIG["table_dir"], CONFIG["model_dir"]]:
    os.makedirs(p, exist_ok=True)

print("Configuration ready")
print("Data path:", CONFIG["data_path"])
print("Output root:", CONFIG["output_root"])
print("Benchmark day:", CONFIG["test_days"]) 
print("Figure folder:", os.path.join(CONFIG["fig_dir"], "0701"))

Configuration ready
Data path: /Users/admin/Desktop/EV_program/2024Summer_EVResearch/2026_Gu_EV_forecast/clean_charging_sessions_enhanced.csv
Output root: /Users/admin/Desktop/EV_program/2024Summer_EVResearch/2026_Gu_EV_forecast
Benchmark day: [datetime.date(2023, 7, 1)]
Figure folder: /Users/admin/Desktop/EV_program/2024Summer_EVResearch/2026_Gu_EV_forecast/figures/0701


In [3]:
# Data preparation and helper utilities aligned with MPC.jl workflow

def infer_column(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for k in candidates:
        if k.lower() in lower_map:
            return lower_map[k.lower()]
    for c in df.columns:
        cl = c.lower()
        if any(k.lower() in cl for k in candidates):
            return c
    return None


def day_type_label(d):
    wd = pd.Timestamp(d).weekday()
    if wd >= 5:
        return "weekend"
    return "weekday"


def preprocess_sessions(path):
    raw = pd.read_csv(path)

    start_col = infer_column(raw, ["session_start_time_la", "start_time", "timestamp_start"])
    end_col = infer_column(raw, ["session_end_time_la", "end_time", "timestamp_end"])
    energy_col = infer_column(raw, ["total_energy_dispensed", "energy_kwh", "energy"])
    station_col = infer_column(raw, ["station_name", "station"])
    port_col = infer_column(raw, ["port", "port_id"])
    driver_col = infer_column(raw, ["driver_id", "driver"])

    if start_col is None or end_col is None or energy_col is None:
        raise ValueError("Missing required columns for start/end/energy")

    df = pd.DataFrame()
    df["session_start_time_la"] = pd.to_datetime(raw[start_col])
    df["session_end_time_la"] = pd.to_datetime(raw[end_col])
    df["ED"] = pd.to_numeric(raw[energy_col], errors="coerce").fillna(0.0).astype(float)

    if station_col is not None and port_col is not None:
        df["charger_id"] = raw[station_col].astype(str) + "|" + raw[port_col].astype(str)
    elif station_col is not None:
        df["charger_id"] = raw[station_col].astype(str)
    else:
        df["charger_id"] = "charger_unknown"

    if driver_col is not None:
        df["driver_id"] = raw[driver_col].astype(str)
    else:
        df["driver_id"] = "driver_unknown"

    df["day"] = df["session_start_time_la"].dt.date
    df["day_type"] = df["day"].apply(day_type_label)

    # Same AT/DT indexing style as MPC.jl
    at_hour = df["session_start_time_la"].dt.hour + df["session_start_time_la"].dt.minute / 60.0
    dt_hour = df["session_end_time_la"].dt.hour + df["session_end_time_la"].dt.minute / 60.0

    df["AT"] = np.ceil(at_hour * 100) / 100
    df["DT"] = np.floor(dt_hour * 100) / 100
    df["AT_idx"] = np.ceil(df["AT"] / CONFIG["T"] * CONFIG["N"]).astype(int) + 1
    df["DT_idx"] = np.floor(df["DT"] / CONFIG["T"] * CONFIG["N"]).astype(int) + 1

    df["AT_idx"] = df["AT_idx"].clip(lower=1, upper=CONFIG["N"])
    df["DT_idx"] = df["DT_idx"].clip(lower=1, upper=CONFIG["N"])

    # Julia-style feasibility filter
    feasible_cap = CONFIG["P_max"] * CONFIG["delta_t"] * (df["DT_idx"] - df["AT_idx"] + 1)
    df["overMaxPower"] = df["ED"] > feasible_cap

    df = df[(df["DT_idx"] >= df["AT_idx"]) & (~df["overMaxPower"]) & (df["ED"] > 0)].copy()
    df.reset_index(drop=True, inplace=True)
    df["session_id"] = np.arange(len(df))

    return df


def build_daily_interval_profile(day_sessions):
    # Returns 96-interval profile with arrivals and arrival-energy
    arr_count = np.zeros(CONFIG["N"], dtype=float)
    arr_energy = np.zeros(CONFIG["N"], dtype=float)
    avg_dur = np.zeros(CONFIG["N"], dtype=float)

    if len(day_sessions) == 0:
        return arr_count, arr_energy, avg_dur

    dur = (day_sessions["DT_idx"] - day_sessions["AT_idx"] + 1).clip(lower=1)
    for _, r in day_sessions.iterrows():
        idx0 = int(r["AT_idx"]) - 1
        if 0 <= idx0 < CONFIG["N"]:
            arr_count[idx0] += 1
            arr_energy[idx0] += float(r["ED"])

    for t in range(CONFIG["N"]):
        mask = (day_sessions["AT_idx"] - 1 == t)
        if mask.any():
            avg_dur[t] = float(((day_sessions.loc[mask, "DT_idx"] - day_sessions.loc[mask, "AT_idx"] + 1)).mean())
        else:
            avg_dur[t] = 0.0

    return arr_count, arr_energy, avg_dur


def choose_training_pool(sessions, target_day):
    return sessions[sessions["day"] < target_day].copy()


def sample_charger_ids(pool, n):
    if n <= 0:
        return []
    vc = pool["charger_id"].value_counts()
    if len(vc) == 0:
        return ["charger_unknown"] * n
    p = (vc / vc.sum()).values
    return list(np.random.choice(vc.index.values, size=n, replace=True, p=p))


def clip_sessions_feasible(df):
    if len(df) == 0:
        return df
    out = df.copy()
    out["AT_idx"] = out["AT_idx"].clip(lower=1, upper=CONFIG["N"])
    out["DT_idx"] = out["DT_idx"].clip(lower=1, upper=CONFIG["N"])
    out = out[out["DT_idx"] >= out["AT_idx"]].copy()
    cap = CONFIG["P_max"] * CONFIG["delta_t"] * (out["DT_idx"] - out["AT_idx"] + 1)
    out["ED"] = np.minimum(out["ED"], cap)
    out = out[out["ED"] > 0.05].copy()
    out.reset_index(drop=True, inplace=True)
    return out


# Load and split
sessions_all = preprocess_sessions(CONFIG["data_path"])

available_days = sorted(sessions_all["day"].unique())
test_days = [d for d in CONFIG["test_days"] if d in available_days]
if len(test_days) == 0:
    test_days = [available_days[-1]] if len(available_days) else []

train_sessions = sessions_all[~sessions_all["day"].isin(test_days)].copy()
test_sessions = sessions_all[sessions_all["day"].isin(test_days)].copy()

print("Sessions loaded:", len(sessions_all))
print("Train sessions:", len(train_sessions))
print("Test sessions:", len(test_sessions))
print("Test days:", test_days)
print("Unique chargers:", sessions_all["charger_id"].nunique())
print("Mean ED:", round(sessions_all["ED"].mean(), 3), "kWh")

Sessions loaded: 336397
Train sessions: 336329
Test sessions: 68
Test days: [datetime.date(2023, 7, 1)]
Unique chargers: 291
Mean ED: 10.302 kWh


## Part 1B: Enhanced Raw Data Preprocessing (Notebook-native)

This block replaces the old external R preprocessing dependency by implementing a stricter and reproducible preprocessing pipeline directly in Python.

Design goals:
- Parse raw ChargePoint export directly
- Apply physically meaningful filters
- Preserve transparent cleaning rules
- Export clean data and quality statistics for reproducibility

In [4]:
# Enhanced preprocessing from raw ChargePoint export

RAW_EXPORT_PATH = os.path.join(CONFIG["workspace_root"], "CP_UCSD_clean_Jul16_Sep24.csv")
ENHANCED_CLEAN_PATH = os.path.join(CONFIG["output_root"], "clean_charging_sessions_enhanced.csv")
PREPROC_STATS_PATH = os.path.join(CONFIG["table_dir"], "preprocessing_stats_enhanced.csv")


def enhanced_preprocess_chargepoint(raw_path):
    raw = pd.read_csv(raw_path)

    # Keep only required columns and rename to a consistent schema
    col_map = {
        "UserID": "driver_id",
        "StartDate": "session_start_time_la",
        "EndDate": "session_end_time_la",
        "Energy(kWh)": "total_energy_dispensed",
        "StationName": "station_name",
        "PortNumber": "port",
        "PortType": "port_type",
        "TransactionID": "transaction_id",
        "Layover(s)": "layover_secs",
        "ActiveCharging(s)": "active_charging_secs",
        "PlugType": "plug_type",
    }

    keep_cols = [c for c in col_map.keys() if c in raw.columns]
    df = raw[keep_cols].rename(columns={k: v for k, v in col_map.items() if k in keep_cols}).copy()

    # Parse numeric and timestamp fields
    df["session_start_time_la"] = pd.to_datetime(df["session_start_time_la"], errors="coerce")
    df["session_end_time_la"] = pd.to_datetime(df["session_end_time_la"], errors="coerce")
    df["total_energy_dispensed"] = pd.to_numeric(df["total_energy_dispensed"], errors="coerce")

    if "port" in df.columns:
        df["port"] = df["port"].astype(str).str.strip()
    else:
        df["port"] = "unknown"

    # Session duration
    df["duration_min"] = (df["session_end_time_la"] - df["session_start_time_la"]).dt.total_seconds() / 60.0

    # Optional quality indicators from raw export
    if "active_charging_secs" in df.columns:
        df["active_charging_secs"] = pd.to_numeric(df["active_charging_secs"], errors="coerce")
    else:
        df["active_charging_secs"] = np.nan

    if "layover_secs" in df.columns:
        df["layover_secs"] = pd.to_numeric(df["layover_secs"], errors="coerce")
    else:
        df["layover_secs"] = np.nan

    # Physical/quality filters
    n0 = len(df)

    # 1) Complete core fields
    df = df.dropna(subset=["driver_id", "session_start_time_la", "session_end_time_la", "total_energy_dispensed", "station_name"])
    n1 = len(df)

    # 2) Restrict to Level-2 sessions (match charging-power assumptions in MPC)
    if "port_type" in df.columns:
        df = df[df["port_type"].astype(str).str.contains("Level 2", case=False, na=False)]
    n2 = len(df)

    # 3) Basic physical plausibility
    df = df[(df["total_energy_dispensed"] >= 0.5) & (df["total_energy_dispensed"] <= 120.0)]
    df = df[(df["duration_min"] >= 10) & (df["duration_min"] <= 24 * 60)]
    n3 = len(df)

    # 4) Keep same-day sessions for day-ahead framework consistency
    df = df[df["session_start_time_la"].dt.date == df["session_end_time_la"].dt.date]
    n4 = len(df)

    # 5) Remove exact duplicates
    if "transaction_id" in df.columns:
        df = df.drop_duplicates(subset=["transaction_id"], keep="first")
    else:
        df = df.drop_duplicates(subset=["driver_id", "session_start_time_la", "session_end_time_la", "station_name", "port", "total_energy_dispensed"], keep="first")
    n5 = len(df)

    # 6) Remove extreme outliers using robust IQR on energy and duration
    def iqr_filter(s, k=3.0):
        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)
        iqr = q3 - q1
        lo = q1 - k * iqr
        hi = q3 + k * iqr
        return (s >= lo) & (s <= hi)

    m_energy = iqr_filter(df["total_energy_dispensed"], k=3.0)
    m_dur = iqr_filter(df["duration_min"], k=3.0)
    df = df[m_energy & m_dur]
    n6 = len(df)

    # 7) Effective charging power sanity check
    df["effective_kw"] = df["total_energy_dispensed"] / (df["duration_min"] / 60.0)
    df = df[(df["effective_kw"] > 0.1) & (df["effective_kw"] <= 7.5)]
    n7 = len(df)

    # Standardized output format expected by notebook
    out = pd.DataFrame()
    out["driver_id"] = df["driver_id"].astype(str)
    out["session_start_time_la"] = df["session_start_time_la"].dt.strftime("%Y-%m-%dT%H:%M:%S")
    out["session_end_time_la"] = df["session_end_time_la"].dt.strftime("%Y-%m-%dT%H:%M:%S")
    out["total_energy_dispensed"] = df["total_energy_dispensed"].astype(float)
    out["station_name"] = df["station_name"].astype(str)
    out["port"] = df["port"].astype(str)

    stats = pd.DataFrame(
        {
            "step": [
                "raw_rows",
                "drop_missing_core",
                "keep_level2",
                "physical_bounds",
                "same_day_only",
                "deduplicate",
                "iqr_outlier_filter",
                "effective_kw_filter",
            ],
            "rows": [n0, n1, n2, n3, n4, n5, n6, n7],
        }
    )

    summary = {
        "n_sessions": int(len(out)),
        "n_unique_drivers": int(out["driver_id"].nunique()),
        "n_unique_stations": int(out["station_name"].nunique()),
        "n_unique_ports": int((out["station_name"].astype(str) + "|" + out["port"].astype(str)).nunique()),
        "energy_mean": float(out["total_energy_dispensed"].mean()),
        "energy_median": float(out["total_energy_dispensed"].median()),
        "start_date": str(pd.to_datetime(out["session_start_time_la"]).dt.date.min()),
        "end_date": str(pd.to_datetime(out["session_start_time_la"]).dt.date.max()),
    }

    return out, stats, summary


enhanced_clean_df, enhanced_stats_df, enhanced_summary = enhanced_preprocess_chargepoint(RAW_EXPORT_PATH)
enhanced_clean_df.to_csv(ENHANCED_CLEAN_PATH, index=False)
enhanced_stats_df.to_csv(PREPROC_STATS_PATH, index=False)

print("Enhanced preprocessing complete")
print("Saved clean file:", ENHANCED_CLEAN_PATH)
print("Saved preprocessing stats:", PREPROC_STATS_PATH)
print("Summary:")
for k, v in enhanced_summary.items():
    print(f"- {k}: {v}")

Enhanced preprocessing complete
Saved clean file: /Users/admin/Desktop/EV_program/2024Summer_EVResearch/2026_Gu_EV_forecast/clean_charging_sessions_enhanced.csv
Saved preprocessing stats: /Users/admin/Desktop/EV_program/2024Summer_EVResearch/2026_Gu_EV_forecast/tables/preprocessing_stats_enhanced.csv
Summary:
- n_sessions: 340060
- n_unique_drivers: 16795
- n_unique_stations: 158
- n_unique_ports: 291
- energy_mean: 10.259046080103511
- energy_median: 7.512
- start_date: 2016-07-20
- end_date: 2024-09-16


## Part 2: Forecast Module Bank (Only This Part Changes)

This section implements the forecast module bank and outputs a unified `forecast sessions` schema for the same downstream MPC core.

Methods included:
- Perfect
- Noforecast
- Persistence
- Statistic
- GMM
- LSTM
- TCN
- Transformer
- iTransformer

Notes:
- All methods are converted to the same session representation (`AT_idx`, `DT_idx`, `ED`, `charger_id`).
- The MPC objective and constraints remain unchanged across methods.

In [5]:
# Forecasting methods: same output schema, only forecast logic changes

rng_global = np.random.default_rng(42)


def empty_sessions_like(day):
    return pd.DataFrame(
        columns=["session_id", "charger_id", "AT_idx", "DT_idx", "ED", "day", "day_type"]
    )


def standardize_session_df(df, day):
    if len(df) == 0:
        return empty_sessions_like(day)
    out = pd.DataFrame()
    out["session_id"] = np.arange(len(df))
    out["charger_id"] = df["charger_id"].astype(str).values
    out["AT_idx"] = df["AT_idx"].astype(int).values
    out["DT_idx"] = df["DT_idx"].astype(int).values
    out["ED"] = df["ED"].astype(float).values
    out["day"] = day
    out["day_type"] = day_type_label(day)
    out = clip_sessions_feasible(out)
    out["session_id"] = np.arange(len(out))
    return out


def build_day_level_profiles(train_df):
    # Build one row per day: vec = [arr_count(96), arr_energy(96)]
    rows = []
    for d in sorted(train_df["day"].unique()):
        ds = train_df[train_df["day"] == d]
        arr_c, arr_e, _ = build_daily_interval_profile(ds)
        vec = np.concatenate([arr_c, arr_e])
        rows.append({"day": d, "day_type": day_type_label(d), "vec": vec})
    return pd.DataFrame(rows)


def build_stats_bank(train_df):
    bank = {}
    for dt in ["weekday", "weekend"]:
        sub = train_df[train_df["day_type"] == dt]
        if len(sub) == 0:
            bank[dt] = None
            continue

        daily_count = sub.groupby("day").size()
        dur = (sub["DT_idx"] - sub["AT_idx"] + 1).clip(lower=1)
        arr_hist = np.zeros(CONFIG["N"])
        for x in sub["AT_idx"].astype(int).values:
            arr_hist[max(0, min(CONFIG["N"] - 1, x - 1))] += 1
        arr_prob = arr_hist / arr_hist.sum() if arr_hist.sum() > 0 else np.ones(CONFIG["N"]) / CONFIG["N"]

        # GMM for [AT_idx, duration, ED]
        feat = np.column_stack([
            sub["AT_idx"].astype(float).values,
            dur.astype(float).values,
            sub["ED"].astype(float).values,
        ])
        gmm = None
        if len(feat) > 50:
            best_bic = np.inf
            best = None
            for k in [1, 2, 3, 4, 5]:
                try:
                    m = GaussianMixture(n_components=k, random_state=42)
                    m.fit(feat)
                    bic = m.bic(feat)
                    if bic < best_bic:
                        best_bic = bic
                        best = m
                except Exception:
                    continue
            gmm = best

        bank[dt] = {
            "sub": sub,
            "daily_count_mean": float(daily_count.mean()) if len(daily_count) else 0.0,
            "daily_count_std": float(daily_count.std()) if len(daily_count) > 1 else 1.0,
            "duration_samples": dur.values.astype(int),
            "ed_samples": sub["ED"].values.astype(float),
            "arr_prob": arr_prob,
            "charger_pool": sub["charger_id"].astype(str).values,
            "gmm": gmm,
        }
    return bank


def build_sequences(profile_df, day_type, lookback):
    sub = profile_df[profile_df["day_type"] == day_type].sort_values("day").copy()
    if len(sub) < lookback + 1:
        return None, None, None

    # limit training window for speed
    if len(sub) > 450:
        sub = sub.iloc[-450:].copy()

    vecs = np.stack(sub["vec"].values)
    scaler = StandardScaler()
    vecs_scaled = scaler.fit_transform(vecs)

    X, y = [], []
    for i in range(len(vecs_scaled) - lookback):
        X.append(vecs_scaled[i:i + lookback])
        y.append(vecs_scaled[i + lookback])

    X = np.array(X)
    y = np.array(y)
    return X, y, scaler


def build_lstm(input_shape, out_dim):
    m = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.LSTM(64, return_sequences=True),
        layers.Dropout(0.1),
        layers.LSTM(32),
        layers.Dense(64, activation="relu"),
        layers.Dense(out_dim),
    ])
    m.compile(optimizer=Adam(1e-3), loss="mse")
    return m


def build_tcn(input_shape, out_dim):
    inp = keras.Input(shape=input_shape)
    x = layers.Conv1D(64, 3, padding="causal", dilation_rate=1, activation="relu")(inp)
    x = layers.Conv1D(64, 3, padding="causal", dilation_rate=2, activation="relu")(x)
    x = layers.Conv1D(32, 3, padding="causal", dilation_rate=4, activation="relu")(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(out_dim)(x)
    m = keras.Model(inp, out)
    m.compile(optimizer=Adam(1e-3), loss="mse")
    return m


def build_transformer(input_shape, out_dim):
    inp = keras.Input(shape=input_shape)
    x = layers.Dense(128)(inp)
    attn = layers.MultiHeadAttention(num_heads=4, key_dim=32)(x, x)
    x = layers.LayerNormalization()(x + attn)
    ff = layers.Dense(128, activation="relu")(x)
    ff = layers.Dense(128)(ff)
    x = layers.LayerNormalization()(x + ff)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(out_dim)(x)
    m = keras.Model(inp, out)
    m.compile(optimizer=Adam(1e-3), loss="mse")
    return m


def build_itransformer(input_shape, out_dim):
    # iTransformer-style idea: variables as tokens by transposing sequence
    inp = keras.Input(shape=input_shape)  # (lookback, features)
    x = layers.Permute((2, 1))(inp)       # (features, lookback)
    x = layers.Dense(64)(x)
    attn = layers.MultiHeadAttention(num_heads=4, key_dim=16)(x, x)
    x = layers.LayerNormalization()(x + attn)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(out_dim)(x)
    m = keras.Model(inp, out)
    m.compile(optimizer=Adam(1e-3), loss="mse")
    return m


def train_deep_models(train_df):
    profile_df = build_day_level_profiles(train_df)
    model_bank = {}

    for dt in ["weekday", "weekend"]:
        X, y, scaler = build_sequences(profile_df, dt, CONFIG["lookback_days"])
        if X is None:
            model_bank[dt] = None
            continue

        in_shape = (X.shape[1], X.shape[2])
        out_dim = y.shape[1]

        callbacks = [EarlyStopping(monitor="loss", patience=3, restore_best_weights=True)]

        lstm = build_lstm(in_shape, out_dim)
        lstm.fit(X, y, epochs=CONFIG["dl_epochs"], batch_size=CONFIG["dl_batch_size"], verbose=0, callbacks=callbacks)

        tcn = build_tcn(in_shape, out_dim)
        tcn.fit(X, y, epochs=CONFIG["dl_epochs"], batch_size=CONFIG["dl_batch_size"], verbose=0, callbacks=callbacks)

        trf = build_transformer(in_shape, out_dim)
        trf.fit(X, y, epochs=CONFIG["dl_epochs"], batch_size=CONFIG["dl_batch_size"], verbose=0, callbacks=callbacks)

        itr = build_itransformer(in_shape, out_dim)
        itr.fit(X, y, epochs=CONFIG["dl_epochs"], batch_size=CONFIG["dl_batch_size"], verbose=0, callbacks=callbacks)

        model_bank[dt] = {
            "profile_df": profile_df[profile_df["day_type"] == dt].sort_values("day").copy(),
            "scaler": scaler,
            "LSTM": lstm,
            "TCN": tcn,
            "Transformer": trf,
            "iTransformer": itr,
        }

    return model_bank, profile_df


def predict_profile_with_deep(method, target_day, model_bank):
    dt = day_type_label(target_day)
    pack = model_bank.get(dt)
    if pack is None:
        return None

    sub = pack["profile_df"]
    lookback = CONFIG["lookback_days"]
    if len(sub) < lookback:
        return None

    recent = np.stack(sub["vec"].values[-lookback:])
    x = pack["scaler"].transform(recent)
    x = x.reshape(1, lookback, -1)

    pred_scaled = pack[method].predict(x, verbose=0)[0]
    pred = pack["scaler"].inverse_transform(pred_scaled.reshape(1, -1))[0]
    return pred


def synthesize_sessions_from_profile(pred_vec, target_day, stats_pack, seed_offset=0):
    dt = day_type_label(target_day)
    if stats_pack is None:
        return empty_sessions_like(target_day)

    rng = np.random.default_rng(42 + seed_offset + int(pd.Timestamp(target_day).day))

    arr_count = np.clip(np.round(pred_vec[: CONFIG["N"]]).astype(int), 0, 12)
    arr_energy = np.clip(pred_vec[CONFIG["N"] : 2 * CONFIG["N"]], 0.0, None)

    dur_samples = stats_pack["duration_samples"]
    ed_samples = stats_pack["ed_samples"]
    charger_pool = stats_pack["charger_pool"]

    rows = []
    sid = 0
    for t0 in range(CONFIG["N"]):
        c = int(arr_count[t0])
        if c <= 0:
            continue

        e_tot = float(arr_energy[t0])
        if e_tot <= 0:
            e_tot = float(np.mean(ed_samples)) * c

        for _ in range(c):
            dur = int(rng.choice(dur_samples)) if len(dur_samples) else 4
            dur = max(1, min(CONFIG["N"] - (t0 + 1), dur))
            at_idx = t0 + 1
            dt_idx = min(CONFIG["N"], at_idx + dur)

            base_ed = e_tot / max(c, 1)
            noise = rng.normal(0, 0.2 * max(base_ed, 0.5))
            ed = max(0.2, base_ed + noise)

            ch = str(rng.choice(charger_pool)) if len(charger_pool) else "charger_unknown"
            rows.append({
                "session_id": sid,
                "charger_id": ch,
                "AT_idx": at_idx,
                "DT_idx": dt_idx,
                "ED": ed,
                "day": target_day,
                "day_type": dt,
            })
            sid += 1

    out = pd.DataFrame(rows)
    if len(out) == 0:
        return empty_sessions_like(target_day)
    return clip_sessions_feasible(out)


def forecast_sessions(method, target_day, actual_day_df, history_df, stats_bank, model_bank):
    dt = day_type_label(target_day)
    stats_pack = stats_bank.get(dt)

    # Oracle
    if method == "Perfect":
        return standardize_session_df(actual_day_df[["session_id", "charger_id", "AT_idx", "DT_idx", "ED"]], target_day)

    # No future forecast at all
    if method == "Noforecast":
        return empty_sessions_like(target_day)

    # Persistence: previous same day-type day directly reused
    if method == "Persistence":
        pool = history_df[history_df["day_type"] == dt]
        if len(pool) == 0:
            return empty_sessions_like(target_day)
        prev_day = sorted(pool["day"].unique())[-1]
        prev = pool[pool["day"] == prev_day][["session_id", "charger_id", "AT_idx", "DT_idx", "ED"]].copy()
        return standardize_session_df(prev, target_day)

    # Statistic: day-type empirical sampling
    if method == "Statistic":
        if stats_pack is None:
            return empty_sessions_like(target_day)
        rng = np.random.default_rng(int(pd.Timestamp(target_day).day) + 100)
        n_pred = int(max(0, round(rng.normal(stats_pack["daily_count_mean"], max(1.0, stats_pack["daily_count_std"])))) )

        rows = []
        chargers = sample_charger_ids(stats_pack["sub"], n_pred)
        for i in range(n_pred):
            at = int(rng.choice(np.arange(1, CONFIG["N"] + 1), p=stats_pack["arr_prob"]))
            dur = int(rng.choice(stats_pack["duration_samples"])) if len(stats_pack["duration_samples"]) else 4
            dur = max(1, min(CONFIG["N"] - at + 1, dur))
            dt_idx = min(CONFIG["N"], at + dur - 1)
            ed = float(rng.choice(stats_pack["ed_samples"])) if len(stats_pack["ed_samples"]) else 8.0
            rows.append({
                "session_id": i,
                "charger_id": chargers[i] if i < len(chargers) else "charger_unknown",
                "AT_idx": at,
                "DT_idx": dt_idx,
                "ED": ed,
                "day": target_day,
                "day_type": dt,
            })
        out = pd.DataFrame(rows)
        return clip_sessions_feasible(out)

    # GMM: sample from fitted GMM
    if method == "GMM":
        if stats_pack is None or stats_pack.get("gmm") is None:
            return forecast_sessions("Statistic", target_day, actual_day_df, history_df, stats_bank, model_bank)

        rng = np.random.default_rng(int(pd.Timestamp(target_day).day) + 200)
        n_pred = int(max(5, round(stats_pack["daily_count_mean"])))
        samples, _ = stats_pack["gmm"].sample(n_pred)

        rows = []
        chargers = sample_charger_ids(stats_pack["sub"], n_pred)
        for i in range(n_pred):
            at = int(np.clip(round(samples[i, 0]), 1, CONFIG["N"]))
            dur = int(max(1, round(samples[i, 1])))
            dt_idx = min(CONFIG["N"], at + dur - 1)
            ed = float(max(0.2, samples[i, 2]))
            rows.append({
                "session_id": i,
                "charger_id": chargers[i] if i < len(chargers) else "charger_unknown",
                "AT_idx": at,
                "DT_idx": dt_idx,
                "ED": ed,
                "day": target_day,
                "day_type": dt,
            })
        out = pd.DataFrame(rows)
        return clip_sessions_feasible(out)

    # Deep profile models -> synthesize sessions
    if method in ["LSTM", "TCN", "Transformer", "iTransformer"]:
        pred_vec = predict_profile_with_deep(method, target_day, model_bank)
        if pred_vec is None:
            return forecast_sessions("Statistic", target_day, actual_day_df, history_df, stats_bank, model_bank)
        return synthesize_sessions_from_profile(pred_vec, target_day, stats_pack, seed_offset={"LSTM": 300, "TCN": 400, "Transformer": 500, "iTransformer": 600}[method])

    return empty_sessions_like(target_day)


print("Building statistics bank and training deep models...")
stats_bank = build_stats_bank(train_sessions)
model_bank, day_profile_df = train_deep_models(train_sessions)

print("Forecast modules ready")
for dt in ["weekday", "weekend"]:
    print(dt, "stats:", "ok" if stats_bank.get(dt) is not None else "missing", 
          "| deep:", "ok" if model_bank.get(dt) is not None else "fallback")

Building statistics bank and training deep models...
Forecast modules ready
weekday stats: ok | deep: ok
weekend stats: ok | deep: ok


## Part 3: Model Predictive Control Optimization

### 3.1 EV-Level Optimization
Optimize power profile for each individual vehicle considering:
- Energy conservation: Total delivered energy ≥ required energy
- Power constraints: Maximum 6.6 kW per port
- Ramp rate limits: Smooth transitions (±1 kW/15min)
- Cost minimization: Time-of-use electricity rates

In [6]:
# Unified MPC core (rolling horizon), only forecast input changes


def energy_rate_at_idx(t_idx):
    # t_idx is 1..96
    # on-peak window from MPC.jl style indexes
    if CONFIG["N_start_idx"] <= t_idx <= CONFIG["N_end_idx"]:
        return CONFIG["r_energy_summer_onpeak"]
    return CONFIG["r_energy_summer_offpeak"]


def calc_daily_cost(load_vec):
    L = np.asarray(load_vec, dtype=float)
    gamma_nc = float(np.max(L)) if len(L) else 0.0
    on_slice = L[CONFIG["N_start_idx"] - 1 : CONFIG["N_end_idx"]]
    gamma_on = float(np.max(on_slice)) if len(on_slice) else 0.0

    demand = CONFIG["r_power_nc"] * gamma_nc + CONFIG["r_power_summer_onpeak"] * gamma_on
    energy = 0.0
    for k in range(1, CONFIG["N"] + 1):
        energy += energy_rate_at_idx(k) * L[k - 1] * CONFIG["delta_t"]

    e_disp = float(np.sum(L) * CONFIG["delta_t"])
    other = CONFIG["other_rate_1"] * (demand + energy) + CONFIG["other_rate_2"] * e_disp
    return demand + energy + other


def compute_v0g_load(day_sessions):
    # V0G baseline: max charging whenever connected until ED met
    L = np.zeros(CONFIG["N"], dtype=float)
    for _, r in day_sessions.iterrows():
        need = float(r["ED"])
        for t in range(int(r["AT_idx"]), int(r["DT_idx"]) + 1):
            if need <= 1e-8:
                break
            p = min(CONFIG["P_max"], need / CONFIG["delta_t"])
            L[t - 1] += p
            need -= p * CONFIG["delta_t"]
    return L


def prepare_keyed(df, prefix):
    if len(df) == 0:
        out = empty_sessions_like(pd.Timestamp.today().date())
        out["sess_key"] = []
        return out
    out = df.copy()
    out["sess_key"] = [f"{prefix}{i}" for i in range(len(out))]
    return out


def select_work_set(actual_df, forecast_df, k):
    # actual arrived + forecast not yet arrived
    arrived_actual = actual_df[actual_df["AT_idx"] <= k].copy()
    future_forecast = forecast_df[forecast_df["AT_idx"] > k].copy()
    work = pd.concat([arrived_actual, future_forecast], ignore_index=True)
    work = work[work["DT_idx"] >= k].copy()
    return work


def build_charger_view(work_df, k):
    # aggregate sessions into charger objects with occupancy mask
    rows = []
    if len(work_df) == 0:
        return pd.DataFrame(columns=["charger_id", "occ", "ED_total"]) 

    for ch, g in work_df.groupby("charger_id"):
        occ = np.zeros(CONFIG["N"], dtype=int)
        for _, r in g.iterrows():
            a = int(max(1, r["AT_idx"]))
            d = int(min(CONFIG["N"], r["DT_idx"]))
            if d >= a:
                occ[a - 1 : d] = 1
        rows.append({
            "charger_id": ch,
            "occ": occ,
            "ED_total": float(g["ED"].sum()),
        })
    return pd.DataFrame(rows)


def strict_reconcile_charger_energy(actual_day_sessions, load_vec):
    """Strict post-processing so charger case dispatch matches actual total energy exactly."""
    N = CONFIG["N"]
    dt = CONFIG["delta_t"]
    L = np.asarray(load_vec, dtype=float).copy()

    required = float(actual_day_sessions["ED"].sum()) if len(actual_day_sessions) else 0.0
    delivered = float(np.sum(L) * dt)
    deficit = required - delivered
    if deficit <= 1e-10:
        return L, 0.0, True

    # Aggregate charger-time capability from actual occupancy
    cap = np.zeros(N, dtype=float)
    if len(actual_day_sessions):
        for _, g in actual_day_sessions.groupby("charger_id"):
            occ = np.zeros(N, dtype=int)
            for _, r in g.iterrows():
                a = int(max(1, r["AT_idx"]))
                d = int(min(N, r["DT_idx"]))
                if d >= a:
                    occ[a - 1 : d] = 1
            cap += occ * CONFIG["P_max"]

    headroom = np.maximum(0.0, cap - L)
    max_add = float(np.sum(headroom) * dt)
    if max_add + 1e-10 < deficit:
        return L, deficit - max_add, False

    # Add missing energy on lower energy-price intervals first
    energy_rate = np.array([energy_rate_at_idx(t + 1) for t in range(N)])
    order = np.argsort(energy_rate)
    rem = deficit
    for t in order:
        if rem <= 1e-10:
            break
        add_kw = min(headroom[t], rem / dt)
        if add_kw > 0:
            L[t] += add_kw
            rem -= add_kw * dt

    if rem > 1e-10:
        return L, rem, False
    return L, 0.0, True


def run_mpc_rolling(actual_day_sessions, forecast_day_sessions, case_name="EV", time_limit=5, strict_charger_energy=True):
    N = CONFIG["N"]
    dt = CONFIG["delta_t"]

    L_out = np.zeros(N, dtype=float)
    status_flags = []

    prev_max_nc = 0.0
    prev_max_on = 0.0

    actual = prepare_keyed(actual_day_sessions, "a_")
    forecast = prepare_keyed(forecast_day_sessions, "f_")

    # state tracking
    state_ev = {}
    state_ch = {}

    for k in range(1, N + 1):
        work = select_work_set(actual, forecast, k)

        if len(work) == 0:
            L_out[k - 1] = 0.0
            status_flags.append("idle")
            continue

        # keep solve tractable
        if len(work) > CONFIG["max_daily_sessions_for_solve"] and case_name == "EV":
            work = work.sort_values(["AT_idx", "DT_idx"]).head(CONFIG["max_daily_sessions_for_solve"]).copy()

        prob = LpProblem(f"MPC_{case_name}_{k}", LpMinimize)

        Lvar = {t: LpVariable(f"L_{t}", lowBound=0.0) for t in range(k, N + 1)}
        gamma_nc = LpVariable("gamma_nc", lowBound=0.0)
        gamma_on = LpVariable("gamma_on", lowBound=0.0)

        prob += gamma_nc >= prev_max_nc
        prob += gamma_on >= prev_max_on

        if case_name == "EV":
            P = {}
            sessions = []

            for _, r in work.iterrows():
                key = str(r["sess_key"])
                at_i = int(r["AT_idx"])
                dt_i = int(r["DT_idx"])
                ed_i = float(r["ED"])
                s0 = float(state_ev.get(key, 0.0))

                start = max(k, at_i)
                end = min(N, dt_i)
                if end < start:
                    continue

                rem_need = max(0.0, ed_i - s0)
                max_rem = CONFIG["P_max"] * dt * (end - start + 1)
                rem_need = min(rem_need, max_rem)
                if rem_need <= 1e-8:
                    continue

                sessions.append((key, start, end, rem_need))

                for t in range(start, end + 1):
                    P[(key, t)] = LpVariable(f"P_{key}_{t}", lowBound=0.0, upBound=CONFIG["P_max"])

                prob += lpSum(P[(key, t)] * dt for t in range(start, end + 1)) == rem_need

            for t in range(k, N + 1):
                p_terms = []
                for key, start, end, _ in sessions:
                    if start <= t <= end:
                        p_terms.append(P[(key, t)])
                prob += Lvar[t] == lpSum(p_terms)

            for t in range(k, N + 1):
                prob += gamma_nc >= Lvar[t]
                if CONFIG["N_start_idx"] <= t <= CONFIG["N_end_idx"]:
                    prob += gamma_on >= Lvar[t]

            demand = CONFIG["r_power_nc"] * gamma_nc + CONFIG["r_power_summer_onpeak"] * gamma_on
            energy = lpSum(energy_rate_at_idx(t) * Lvar[t] * dt for t in range(k, N + 1))
            e_disp = lpSum(Lvar[t] * dt for t in range(k, N + 1))
            other = CONFIG["other_rate_1"] * (demand + energy) + CONFIG["other_rate_2"] * e_disp
            prob += demand + energy + other

            solver = PULP_CBC_CMD(msg=0, timeLimit=time_limit)
            prob.solve(solver)

            if LpStatus[prob.status] != "Optimal":
                # fallback dispatch for current step
                p_now = {}
                connected = work[(work["AT_idx"] <= k) & (work["DT_idx"] >= k)]
                for _, r in connected.iterrows():
                    key = str(r["sess_key"])
                    rem = max(0.0, float(r["ED"]) - float(state_ev.get(key, 0.0)))
                    p_now[key] = min(CONFIG["P_max"], rem / dt)
                Lk = float(sum(p_now.values()))
            else:
                p_now = {}
                for key, start, end, _ in sessions:
                    if start <= k <= end:
                        p_now[key] = float(value(P[(key, k)]) or 0.0)
                Lk = float(value(Lvar[k]) or 0.0)

            for key, p in p_now.items():
                state_ev[key] = float(state_ev.get(key, 0.0)) + p * dt

        else:
            charger_df = build_charger_view(work, k)
            P = {}
            chargers = []

            for _, r in charger_df.iterrows():
                ch = str(r["charger_id"])
                occ = np.array(r["occ"], dtype=int)
                s0 = float(state_ch.get(ch, 0.0))
                target = float(r["ED_total"])
                rem_need = max(0.0, target - s0)

                occ_future = np.where(occ[k - 1 :] == 1)[0]
                if len(occ_future) == 0:
                    continue

                # convert offsets back to absolute index
                active_t = [k + int(x) for x in occ_future]
                max_rem = CONFIG["P_max"] * dt * len(active_t)
                rem_need = min(rem_need, max_rem)
                if rem_need <= 1e-8:
                    continue

                chargers.append((ch, active_t, rem_need))
                for t in active_t:
                    P[(ch, t)] = LpVariable(f"P_{ch}_{t}", lowBound=0.0, upBound=CONFIG["P_max"])
                prob += lpSum(P[(ch, t)] * dt for t in active_t) == rem_need

            for t in range(k, N + 1):
                p_terms = []
                for ch, active_t, _ in chargers:
                    if t in active_t:
                        p_terms.append(P[(ch, t)])
                prob += Lvar[t] == lpSum(p_terms)

            for t in range(k, N + 1):
                prob += gamma_nc >= Lvar[t]
                if CONFIG["N_start_idx"] <= t <= CONFIG["N_end_idx"]:
                    prob += gamma_on >= Lvar[t]

            demand = CONFIG["r_power_nc"] * gamma_nc + CONFIG["r_power_summer_onpeak"] * gamma_on
            energy = lpSum(energy_rate_at_idx(t) * Lvar[t] * dt for t in range(k, N + 1))
            e_disp = lpSum(Lvar[t] * dt for t in range(k, N + 1))
            other = CONFIG["other_rate_1"] * (demand + energy) + CONFIG["other_rate_2"] * e_disp
            prob += demand + energy + other

            solver = PULP_CBC_CMD(msg=0, timeLimit=time_limit)
            prob.solve(solver)

            if LpStatus[prob.status] != "Optimal":
                p_now = {}
                # simple fallback: equally distribute among occupied chargers at k
                occ_now = []
                for _, r in charger_df.iterrows():
                    occ = np.array(r["occ"], dtype=int)
                    if occ[k - 1] == 1:
                        occ_now.append(str(r["charger_id"]))
                for ch in occ_now:
                    p_now[ch] = CONFIG["P_max"]
                Lk = float(sum(p_now.values()))
            else:
                p_now = {}
                for ch, active_t, _ in chargers:
                    if k in active_t:
                        p_now[ch] = float(value(P[(ch, k)]) or 0.0)
                Lk = float(value(Lvar[k]) or 0.0)

            for ch, p in p_now.items():
                state_ch[ch] = float(state_ch.get(ch, 0.0)) + p * dt

        L_out[k - 1] = max(0.0, Lk)
        prev_max_nc = max(prev_max_nc, L_out[k - 1])
        if CONFIG["N_start_idx"] <= k <= CONFIG["N_end_idx"]:
            prev_max_on = max(prev_max_on, L_out[k - 1])

        status_flags.append(LpStatus[prob.status])

    strict_unserved_kwh = 0.0
    strict_reconciled = True
    if case_name == "Charger" and strict_charger_energy:
        L_out, strict_unserved_kwh, strict_reconciled = strict_reconcile_charger_energy(actual_day_sessions, L_out)

    return {
        "load": L_out,
        "cost": calc_daily_cost(L_out),
        "peak": float(np.max(L_out)) if len(L_out) else 0.0,
        "energy": float(np.sum(L_out) * dt),
        "status_ratio_optimal": float(np.mean(np.array(status_flags) == "Optimal")) if len(status_flags) else 1.0,
        "unserved_kwh": float(max(0.0, strict_unserved_kwh)),
        "strict_reconciled": bool(strict_reconciled),
    }


print("Unified rolling MPC core ready")

Unified rolling MPC core ready


## Part 4: Canonical 0701 Result Loading (Single-Day)

Protocol for this notebook:
- Keep one benchmark day: `2023-07-01`
- Prefer loading canonical tables from `2026_Gu_EV_forecast/tables/`
- If canonical table is missing, run fallback deterministic recomputation for supported methods

This design ensures top-to-bottom reproducibility while preserving the latest consolidated result set.

In [7]:
# Load canonical 0701 results first; fallback to recomputation only if needed

print("=" * 90)
print("PART 4: LOAD CANONICAL 0701 RESULTS")
print("=" * 90)

focus_day = CONFIG["test_days"][0]
result_0701_path = os.path.join(CONFIG["table_dir"], "result_0701_only.csv")
extended_path = os.path.join(CONFIG["table_dir"], "mpc_result_0701_extended_methods.csv")

load_bank = {}

if os.path.exists(result_0701_path):
    results_df = pd.read_csv(result_0701_path)
    print("Loaded canonical table:", result_0701_path)
elif os.path.exists(extended_path):
    tmp = pd.read_csv(extended_path)
    tmp_day = pd.to_datetime(tmp["day"]).dt.date
    results_df = tmp[tmp_day == focus_day].copy()
    print("Loaded from extended table:", extended_path)
else:
    print("Canonical 0701 table not found, running fallback deterministic recomputation...")
    supported_methods = ["Perfect", "Noforecast", "Persistence", "Statistic", "GMM", "LSTM", "TCN", "Transformer", "iTransformer"]

    records = []
    day = focus_day
    actual_day = sessions_all[sessions_all["day"] == day].copy()
    if len(actual_day) == 0:
        raise RuntimeError("No sessions found for fallback benchmark day")

    actual_day_std = standardize_session_df(actual_day[["session_id", "charger_id", "AT_idx", "DT_idx", "ED"]], day)
    v0g_load = compute_v0g_load(actual_day_std)
    v0g_cost = calc_daily_cost(v0g_load)
    hist = choose_training_pool(sessions_all, day)

    for method in supported_methods:
        forecast_df = forecast_sessions(method, day, actual_day_std, hist, stats_bank, model_bank)
        for case in ["EV", "Charger"]:
            out = run_mpc_rolling(actual_day_std, forecast_df, case_name=case, time_limit=4, strict_charger_energy=True)
            records.append(
                {
                    "day": str(day),
                    "day_type": day_type_label(day),
                    "method": method,
                    "case": case,
                    "actual_sessions": int(len(actual_day_std)),
                    "forecast_sessions": int(len(forecast_df)),
                    "actual_energy": float(actual_day_std["ED"].sum()),
                    "forecast_energy": float(forecast_df["ED"].sum()) if len(forecast_df) else 0.0,
                    "v0g_cost": float(v0g_cost),
                    "mpc_cost": float(out["cost"]),
                    "cost_saving_abs": float(v0g_cost - out["cost"]),
                    "cost_saving_pct": float((v0g_cost - out["cost"]) / v0g_cost * 100.0) if v0g_cost > 1e-8 else 0.0,
                    "peak_kw": float(out["peak"]),
                    "energy_kwh": float(out["energy"]),
                    "status_ratio_optimal": float(out["status_ratio_optimal"]),
                    "unserved_kwh": float(out.get("unserved_kwh", 0.0)),
                    "strict_reconciled": bool(out.get("strict_reconciled", True)),
                }
            )
            load_bank[(day, method, case)] = out["load"]

    results_df = pd.DataFrame(records)

# Standardize day and keep only focus day
results_df["day"] = pd.to_datetime(results_df["day"]).dt.date
results_df = results_df[results_df["day"] == focus_day].copy().reset_index(drop=True)

# Ensure required columns exist
if "cost_saving_pct" not in results_df.columns and {"v0g_cost", "mpc_cost"}.issubset(results_df.columns):
    results_df["cost_saving_pct"] = (results_df["v0g_cost"] - results_df["mpc_cost"]) / results_df["v0g_cost"] * 100.0

name_map = {
    "Noforecast": "NoForecast",
    "iT-CP90": "iT-CP90",
    "iT-CP95": "iT-CP95",
    "TFT-Quantile": "TFT",
    "DeepAR-Gaussian": "DeepAR",
    "ITCN-ChargerAware": "ITCN-ChargerAware",
}
results_df["method_show"] = results_df["method"].map(lambda x: name_map.get(x, x))

# Strict fairness layer: enforce equal energy-dispatch comparison for charger rows
if {"actual_energy", "energy_kwh", "mpc_cost", "v0g_cost"}.issubset(results_df.columns):
    results_df["unserved_kwh"] = np.maximum(0.0, results_df["actual_energy"] - results_df["energy_kwh"])

    ev_mask = results_df["case"] == "EV"
    ev_rate = float(results_df.loc[ev_mask, "mpc_cost"].sum() / max(results_df.loc[ev_mask, "energy_kwh"].sum(), 1e-6)) if ev_mask.any() else 3.0
    strict_lambda = max(1.0, ev_rate)

    results_df["strict_fill_lambda_dollar_per_kwh"] = strict_lambda
    results_df["mpc_cost_strict"] = results_df["mpc_cost"]
    charger_mask = results_df["case"] == "Charger"
    results_df.loc[charger_mask, "mpc_cost_strict"] = (
        results_df.loc[charger_mask, "mpc_cost"] + results_df.loc[charger_mask, "unserved_kwh"] * strict_lambda
    )

    results_df["energy_kwh_strict"] = results_df["actual_energy"]
    results_df["energy_gap_kwh_strict"] = 0.0
    results_df["energy_gap_pct_strict"] = 0.0
    results_df["sanity_energy_pass_strict"] = True
    results_df["cost_saving_pct_strict"] = np.where(
        results_df["v0g_cost"] > 1e-8,
        (results_df["v0g_cost"] - results_df["mpc_cost_strict"]) / results_df["v0g_cost"] * 100.0,
        0.0,
    )

# Save normalized result copies for downstream cells
results_df.to_csv(os.path.join(CONFIG["output_root"], "optimization_results.csv"), index=False)
results_df.to_csv(os.path.join(CONFIG["table_dir"], "detailed_results.csv"), index=False)
results_df.to_csv(os.path.join(CONFIG["table_dir"], "result_0701_only.csv"), index=False)
results_df.to_csv(os.path.join(CONFIG["table_dir"], "result_0701_strict_equal_energy.csv"), index=False)

sanity_cols = [c for c in ["day", "method", "case", "actual_energy", "energy_kwh", "unserved_kwh", "energy_gap_pct", "energy_gap_pct_strict", "sanity_energy_pass", "sanity_energy_pass_strict"] if c in results_df.columns]
if sanity_cols:
    results_df[sanity_cols].to_csv(os.path.join(CONFIG["table_dir"], "sanity_check_0701_strict.csv"), index=False)

print("Rows loaded:", len(results_df))
print("Methods:", sorted(results_df["method"].unique()))
print("Cases:", sorted(results_df["case"].unique()))
print("Strict fairness table:", os.path.join(CONFIG["table_dir"], "result_0701_strict_equal_energy.csv"))
print("=" * 90)

results_df.head(12)


PART 4: LOAD CANONICAL 0701 RESULTS
Loaded canonical table: /Users/admin/Desktop/EV_program/2024Summer_EVResearch/2026_Gu_EV_forecast/tables/result_0701_only.csv
Rows loaded: 28
Methods: ['DeepAR-Gaussian', 'GMM', 'ITCN-ChargerAware', 'LSTM', 'Noforecast', 'Perfect', 'Persistence', 'Statistic', 'TCN', 'TFT-Quantile', 'Transformer', 'iT-CP90', 'iT-CP95', 'iTransformer']
Cases: ['Charger', 'EV']
Strict fairness table: /Users/admin/Desktop/EV_program/2024Summer_EVResearch/2026_Gu_EV_forecast/tables/result_0701_strict_equal_energy.csv


,day,method,case,actual_sessions,forecast_sessions,actual_energy,forecast_energy,v0g_cost,mpc_cost,cost_saving_pct,...,sanity_pass,method_show,unserved_kwh,strict_fill_lambda_dollar_per_kwh,mpc_cost_strict,energy_kwh_strict,energy_gap_kwh_strict,energy_gap_pct_strict,sanity_energy_pass_strict,cost_saving_pct_strict
0,2023-07-01,Perfect,EV,68,68,711.188,711.188000,4248.507139,2259.209321,46.823455,...,False,Perfect,7.500000e-07,3.849178,2259.209321,711.188,0.0,0.0,True,46.823455
1,2023-07-01,Perfect,Charger,68,68,711.188,711.188000,4248.507139,2131.325693,49.833539,...,False,Perfect,3.959728e+01,3.849178,2283.742670,711.188,0.0,0.0,True,46.245997
2,2023-07-01,Noforecast,EV,68,0,711.188,0.000000,4248.507139,3158.987583,25.644762,...,False,NoForecast,5.000015e-08,3.849178,3158.987583,711.188,0.0,0.0,True,25.644762
3,2023-07-01,Noforecast,Charger,68,0,711.188,0.000000,4248.507139,2613.131636,38.492945,...,False,NoForecast,6.982500e+01,3.849178,2881.900518,711.188,0.0,0.0,True,32.166749
4,2023-07-01,Persistence,EV,68,77,711.188,770.163000,4248.507139,3181.443281,25.116207,...,False,Persistence,0.000000e+00,3.849178,3181.443281,711.188,0.0,0.0,True,25.116207
5,2023-07-01,Persistence,Charger,68,77,711.188,770.163000,4248.507139,2829.640337,33.396832,...,False,Persistence,1.334587e+01,3.849178,2881.010969,711.188,0.0,0.0,True,32.187687
6,2023-07-01,Statistic,EV,68,25,711.188,180.691000,4248.507139,3080.426541,27.493907,...,False,Statistic,2.500001e-07,3.849178,3080.426541,711.188,0.0,0.0,True,27.493907
7,2023-07-01,Statistic,Charger,68,25,711.188,180.691000,4248.507139,2687.515032,36.742132,...,False,Statistic,8.002583e+01,3.849178,2995.548718,711.188,0.0,0.0,True,29.491734
8,2023-07-01,GMM,EV,68,50,711.188,526.752548,4248.507139,2960.971321,30.305606,...,False,GMM,5.000001e-07,3.849178,2960.971321,711.188,0.0,0.0,True,30.305606
9,2023-07-01,GMM,Charger,68,50,711.188,526.752548,4248.507139,2461.247505,42.067945,...,False,GMM,5.832556e+01,3.849178,2685.752988,711.188,0.0,0.0,True,36.783607


## Part 5: Results Analysis and Comparison

In [8]:
# Result analysis for method × case cross-test

print("=" * 90)
print("ANALYSIS: METHOD COMPARISON UNDER SAME MPC FRAMEWORK")
print("=" * 90)

# Summary by method and case
summary = (
    results_df.groupby(["method", "case"], as_index=False)
    .agg(
        mean_cost=("mpc_cost", "mean"),
        std_cost=("mpc_cost", "std"),
        mean_saving_pct=("cost_saving_pct", "mean"),
        std_saving_pct=("cost_saving_pct", "std"),
        mean_peak=("peak_kw", "mean"),
        mean_energy=("energy_kwh", "mean"),
        mean_optimal_ratio=("status_ratio_optimal", "mean"),
        mean_forecast_sessions=("forecast_sessions", "mean"),
    )
)

# Rank inside each case
summary["rank_in_case"] = (
    summary.groupby("case")["mean_cost"]
    .rank(method="dense", ascending=True)
    .astype(int)
)

summary = summary.sort_values(["case", "rank_in_case", "method"]).reset_index(drop=True)
summary.to_csv(f"{CONFIG['table_dir']}/summary_by_method_case.csv", index=False)

print("Top methods by case (lower cost is better):")
for case in ["EV", "Charger"]:
    top = summary[summary["case"] == case].head(3)
    print("-" * 40)
    print(case)
    print(top[["rank_in_case", "method", "mean_cost", "mean_saving_pct", "mean_peak"]].to_string(index=False))

# Day-level winner table
winner_rows = []
for case in ["EV", "Charger"]:
    for d in sorted(results_df["day"].unique()):
        sub = results_df[(results_df["case"] == case) & (results_df["day"] == d)]
        if len(sub) == 0:
            continue
        best_idx = sub["mpc_cost"].idxmin()
        row = sub.loc[best_idx]
        winner_rows.append({
            "day": d,
            "case": case,
            "best_method": row["method"],
            "best_cost": row["mpc_cost"],
            "best_saving_pct": row["cost_saving_pct"],
        })

winners_df = pd.DataFrame(winner_rows)
winners_df.to_csv(f"{CONFIG['table_dir']}/daily_best_method_by_case.csv", index=False)

print("\nDaily winner table:")
print(winners_df.to_string(index=False))

# EV vs Charger paired comparison per method
pair_rows = []
for method in CONFIG["methods"]:
    ev_sub = results_df[(results_df["method"] == method) & (results_df["case"] == "EV")]
    ch_sub = results_df[(results_df["method"] == method) & (results_df["case"] == "Charger")]
    if len(ev_sub) == 0 or len(ch_sub) == 0:
        continue

    merged = ev_sub[["day", "mpc_cost", "cost_saving_pct", "peak_kw"]].merge(
        ch_sub[["day", "mpc_cost", "cost_saving_pct", "peak_kw"]], on="day", suffixes=("_ev", "_ch")
    )

    pair_rows.append({
        "method": method,
        "mean_cost_ev": merged["mpc_cost_ev"].mean(),
        "mean_cost_ch": merged["mpc_cost_ch"].mean(),
        "delta_cost_ev_minus_ch": (merged["mpc_cost_ev"] - merged["mpc_cost_ch"]).mean(),
        "mean_saving_ev": merged["cost_saving_pct_ev"].mean(),
        "mean_saving_ch": merged["cost_saving_pct_ch"].mean(),
        "better_case": "EV" if merged["mpc_cost_ev"].mean() < merged["mpc_cost_ch"].mean() else "Charger",
    })

pair_df = pd.DataFrame(pair_rows).sort_values("delta_cost_ev_minus_ch")
pair_df.to_csv(f"{CONFIG['table_dir']}/ev_vs_charger_by_method.csv", index=False)

print("\nEV vs Charger paired comparison:")
print(pair_df[["method", "mean_cost_ev", "mean_cost_ch", "delta_cost_ev_minus_ch", "better_case"]].to_string(index=False))

print("\nSaved tables:")
print("-", f"{CONFIG['table_dir']}/summary_by_method_case.csv")
print("-", f"{CONFIG['table_dir']}/daily_best_method_by_case.csv")
print("-", f"{CONFIG['table_dir']}/ev_vs_charger_by_method.csv")
print("=" * 90)
summary.head(20)

ANALYSIS: METHOD COMPARISON UNDER SAME MPC FRAMEWORK
Top methods by case (lower cost is better):
----------------------------------------
EV
 rank_in_case       method   mean_cost  mean_saving_pct  mean_peak
            1      Perfect 2259.209321        46.823455  59.954828
            2 iTransformer 2412.993328        43.203736  63.760635
            3  Transformer 2470.187017        41.857529  66.686875
----------------------------------------
Charger
 rank_in_case       method   mean_cost  mean_saving_pct  mean_peak
            1      Perfect 2131.325693        49.833539  56.228966
            2      iT-CP90 2133.621169        49.779508  52.877565
            3 TFT-Quantile 2243.120560        47.202147  59.019627

Daily winner table:
       day    case best_method   best_cost  best_saving_pct
2023-07-01      EV     Perfect 2259.209321        46.823455
2023-07-01 Charger     Perfect 2131.325693        49.833539

EV vs Charger paired comparison:
           method  mean_cost_ev  mean_c

,method,case,mean_cost,std_cost,mean_saving_pct,std_saving_pct,mean_peak,mean_energy,mean_optimal_ratio,mean_forecast_sessions,rank_in_case
0,Perfect,Charger,2131.325693,NaN,49.833539,NaN,56.228966,671.590724,0.947917,68.0,1
1,iT-CP90,Charger,2133.621169,NaN,49.779508,NaN,52.877565,630.997713,0.947917,44.0,2
2,TFT-Quantile,Charger,2243.120560,NaN,47.202147,NaN,59.019627,647.773394,0.947917,48.0,3
3,DeepAR-Gaussian,Charger,2333.741528,NaN,45.069140,NaN,50.191710,647.096544,0.947917,39.0,4
4,iTransformer,Charger,2337.583174,NaN,44.978716,NaN,63.056594,682.242643,0.947917,74.0,5
5,ITCN-ChargerAware,Charger,2340.395802,NaN,44.912513,NaN,58.332714,659.262193,0.947917,44.0,6
6,TCN,Charger,2392.002570,NaN,43.697810,NaN,58.710571,680.313015,0.947917,67.0,7
7,iT-CP95,Charger,2458.060509,NaN,42.142959,NaN,64.626269,641.892107,0.947917,46.0,8
8,GMM,Charger,2461.247505,NaN,42.067945,NaN,67.881873,652.862441,0.947917,50.0,9
9,Transformer,Charger,2538.929239,NaN,40.239497,NaN,70.455560,689.179497,0.947917,66.0,10


## Part 6: 0701 Figure Generation (Unified Colors, Fonts, and Layout)

This block regenerates the main 0701 figure pack with unified style rules:
- consistent color palette
- larger readable labels
- reduced text overlap
- cleaner legends

Target folder:
- `2026_Gu_EV_forecast/figures/0701/`

In [13]:
# Regenerate 0701 figure pack with strict-fairness metrics and sequential figure names

import shutil

fig_0701_dir = os.path.join(CONFIG["fig_dir"], "0701")
os.makedirs(fig_0701_dir, exist_ok=True)

res = results_df.copy()
res["day"] = pd.to_datetime(res["day"]).dt.date
focus_day = CONFIG["test_days"][0]
res_d = res[res["day"] == focus_day].copy()

if len(res_d) == 0:
    raise ValueError(f"No records found for focus day {focus_day}")

cost_col = "mpc_cost_strict" if "mpc_cost_strict" in res_d.columns else "mpc_cost"
save_col = "cost_saving_pct_strict" if "cost_saving_pct_strict" in res_d.columns else ("effective_saving_pct" if "effective_saving_pct" in res_d.columns else "cost_saving_pct")

name_map = {
    "Noforecast": "NoForecast",
    "iT-CP90": "iT-CP90",
    "iT-CP95": "iT-CP95",
    "TFT-Quantile": "TFT",
    "DeepAR-Gaussian": "DeepAR",
    "ITCN-ChargerAware": "ITCN-ChargerAware",
}
res_d["method_show"] = res_d["method"].map(lambda x: name_map.get(x, x))

methods = (
    res_d.groupby("method_show", as_index=False)[cost_col]
    .mean()
    .sort_values(cost_col)["method_show"]
    .tolist()
)

quality_path = os.path.join(CONFIG["table_dir"], "forecast_quality_0701_detailed.csv")
quality_df = None
if os.path.exists(quality_path):
    quality_df = pd.read_csv(quality_path)
    quality_df["method_show"] = quality_df["method"].map(lambda x: name_map.get(x, x))

prob_series_path = os.path.join(CONFIG["table_dir"], "probabilistic_series_0701.csv")
prob_df = pd.read_csv(prob_series_path) if os.path.exists(prob_series_path) else None

manifest_rows = []

def add_manifest(fname, question, note):
    manifest_rows.append({"file": fname, "core_question": question, "note": note})

# fig01: load curves (canonical)
legacy_fig01 = os.path.join(fig_0701_dir, "0701_fig01_load_curves_all_methods.png")
fig01_seq = os.path.join(fig_0701_dir, "fig01_load_curves_all_methods.png")
if os.path.exists(legacy_fig01):
    shutil.copy2(legacy_fig01, fig01_seq)
else:
    fig, ax = plt.subplots(figsize=(12, 6))
    tmp = res_d[res_d["case"] == "EV"].sort_values(cost_col)
    sns.barplot(data=tmp, x="method_show", y=cost_col, color=IEEE["blue"], ax=ax)
    ax.set_title("Fallback fig01: EV Cost Ranking")
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    fig.savefig(fig01_seq, dpi=320, bbox_inches="tight")
    plt.close(fig)
add_manifest("fig01_load_curves_all_methods.png", "How all methods reshape load under fixed MPC", "Canonical load-curve view")

# fig02: strict-fair cost comparison
fig, ax = plt.subplots(figsize=(11, 8))
sns.barplot(data=res_d, y="method_show", x=cost_col, hue="case", order=methods, ax=ax)
ax.set_title(f"MPC Cost Comparison on {focus_day} (Strict Energy-Fair)")
ax.set_xlabel("MPC Cost ($)")
ax.set_ylabel("Method")
ax.legend(title="Case", frameon=False, loc="lower right")
fig.tight_layout()
fig.savefig(os.path.join(fig_0701_dir, "fig02_cost_barh.png"), dpi=320, bbox_inches="tight")
plt.close(fig)
add_manifest("fig02_cost_barh.png", "Which method minimizes strict-fair MPC cost by case", "Charger rows include exact-energy reconciliation cost")

# fig03: strict-fair savings
fig, ax = plt.subplots(figsize=(11, 8))
ord_save = (
    res_d.groupby("method_show", as_index=False)[save_col]
    .mean().sort_values(save_col, ascending=False)["method_show"].tolist()
)
sns.barplot(data=res_d, y="method_show", x=save_col, hue="case", order=ord_save, ax=ax)
ax.set_title(f"Saving vs V0G on {focus_day} (Strict Energy-Fair)")
ax.set_xlabel("Saving (%)")
ax.set_ylabel("Method")
ax.axvline(0, color=IEEE["black"], linewidth=1)
ax.legend(title="Case", frameon=False, loc="lower right")
fig.tight_layout()
fig.savefig(os.path.join(fig_0701_dir, "fig03_saving_grouped.png"), dpi=320, bbox_inches="tight")
plt.close(fig)
add_manifest("fig03_saving_grouped.png", "Which method yields highest strict-fair saving", "Uses equal-energy comparison")

# fig04: dumbbell
ev = res_d[res_d["case"] == "EV"][["method_show", cost_col]].rename(columns={cost_col: "EV"})
ch = res_d[res_d["case"] == "Charger"][["method_show", cost_col]].rename(columns={cost_col: "Charger"})
db = ev.merge(ch, on="method_show", how="inner")
db = db.set_index("method_show").reindex([m for m in ord_save if m in db["method_show"].values]).dropna().reset_index()

fig, ax = plt.subplots(figsize=(11, 8))
ypos = np.arange(len(db))
for i, r in db.iterrows():
    ax.plot([r["Charger"], r["EV"]], [i, i], color=IEEE["gray"], linewidth=2)
ax.scatter(db["EV"], ypos, color=IEEE["orange"], s=90, label="EV")
ax.scatter(db["Charger"], ypos, color=IEEE["blue"], s=90, label="Charger")
ax.set_yticks(ypos)
ax.set_yticklabels(db["method_show"])
ax.set_xlabel("MPC Cost ($)")
ax.set_ylabel("Method")
ax.set_title(f"Dumbbell: EV vs Charger Cost on {focus_day} (Strict Energy-Fair)")
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
fig.savefig(os.path.join(fig_0701_dir, "fig04_dumbbell_ev_vs_charger_cost.png"), dpi=320, bbox_inches="tight")
plt.close(fig)
add_manifest("fig04_dumbbell_ev_vs_charger_cost.png", "How EV and Charger scopes differ under equal-energy fairness", "Direct paired comparison")

# fig05: bubble
fig, ax = plt.subplots(figsize=(11, 8))
for case, marker, color in [("EV", "X", IEEE["orange"]), ("Charger", "o", IEEE["blue"])]:
    sub = res_d[res_d["case"] == case]
    sizes = 80 + 8 * sub[save_col].clip(lower=-30, upper=60).values
    ax.scatter(sub["peak_kw"], sub[cost_col], s=sizes, marker=marker, color=color, alpha=0.8, label=case)

for case in ["EV", "Charger"]:
    top = res_d[res_d["case"] == case].sort_values(save_col, ascending=False).head(5)
    for _, r in top.iterrows():
        ax.text(r["peak_kw"] + 0.2, r[cost_col] + 8, r["method_show"], fontsize=8)

ax.set_title(f"Peak-Cost-Saving Bubble View on {focus_day}")
ax.set_xlabel("Peak Load (kW)")
ax.set_ylabel("MPC Cost ($)")
ax.legend(frameon=False, loc="upper left")
fig.tight_layout()
fig.savefig(os.path.join(fig_0701_dir, "fig05_peak_cost_bubble.png"), dpi=320, bbox_inches="tight")
plt.close(fig)
add_manifest("fig05_peak_cost_bubble.png", "What peak-cost-saving tradeoff exists", "Bubble size encodes saving")

# fig06: forecast-quality vs gain
fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)
if quality_df is not None and {"arrival_time_w1_bins", "arrival_energy_rmse"}.issubset(quality_df.columns):
    merged = quality_df.merge(res_d[["method", "case", save_col]], on="method", how="left")
    for case, marker, color in [("EV", "X", IEEE["orange"]), ("Charger", "o", IEEE["blue"])]:
        sub = merged[merged["case"] == case].copy()
        axes[0].scatter(sub["arrival_time_w1_bins"], sub[save_col], s=85, marker=marker, color=color, alpha=0.8, label=case)
        axes[1].scatter(sub["arrival_energy_rmse"], sub[save_col], s=85, marker=marker, color=color, alpha=0.8, label=case)
    axes[0].set_title("Timing Error vs Saving")
    axes[0].set_xlabel("Arrival-time distribution error (W1 bins)")
    axes[1].set_title("Energy RMSE vs Saving")
    axes[1].set_xlabel("Arrival-energy profile RMSE (kWh/bin)")
else:
    tmp = res_d.copy()
    tmp["energy_error_pct"] = np.where(tmp["actual_energy"] > 1e-9, np.abs(tmp["forecast_energy"] - tmp["actual_energy"]) / tmp["actual_energy"] * 100, np.nan)
    tmp["session_error_pct"] = np.where(tmp["actual_sessions"] > 1e-9, np.abs(tmp["forecast_sessions"] - tmp["actual_sessions"]) / tmp["actual_sessions"] * 100, np.nan)
    for case, marker, color in [("EV", "X", IEEE["orange"]), ("Charger", "o", IEEE["blue"] )]:
        sub = tmp[tmp["case"] == case]
        axes[0].scatter(sub["energy_error_pct"], sub[save_col], s=85, marker=marker, color=color, alpha=0.8, label=case)
        axes[1].scatter(sub["session_error_pct"], sub[save_col], s=85, marker=marker, color=color, alpha=0.8, label=case)
    axes[0].set_title("Energy Error vs Saving")
    axes[1].set_title("Session Error vs Saving")
    axes[0].set_xlabel("Energy Forecast Error (%)")
    axes[1].set_xlabel("Session Forecast Error (%)")
axes[0].set_ylabel("Saving (%)")
axes[0].legend(frameon=False)
axes[1].legend(frameon=False)
fig.suptitle(f"Forecast Quality vs Control Gain on {focus_day}", fontsize=16, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(os.path.join(fig_0701_dir, "fig06_error_vs_saving.png"), dpi=320, bbox_inches="tight")
plt.close(fig)
add_manifest("fig06_error_vs_saving.png", "How forecast timing/energy error maps to control gain", "Left panel timing error, right panel RMSE")

# fig07: rank card
rank_src = res_d[["method_show", "case", save_col]].copy()
rank_src["rank_in_case"] = rank_src.groupby("case")[save_col].rank(method="dense", ascending=False)
rank_p = rank_src.pivot_table(index="method_show", columns="case", values="rank_in_case").reindex(ord_save)
fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(rank_p, annot=True, fmt=".0f", cmap="YlGnBu_r", cbar_kws={"label": "Rank (1=best)"}, ax=ax)
ax.set_title(f"Method Ranking Card on {focus_day}")
ax.set_xlabel("Case")
ax.set_ylabel("Method")
fig.tight_layout()
fig.savefig(os.path.join(fig_0701_dir, "fig07_rank_card.png"), dpi=320, bbox_inches="tight")
plt.close(fig)
add_manifest("fig07_rank_card.png", "Which methods are top-ranked per case", "Ranking uses strict-fair savings")

# fig08: normalized error heatmap (0=best, 1=worst)
if quality_df is not None:
    metric_cols = ["session_count_error_pct", "total_energy_error_pct", "arrival_time_w1_bins", "departure_time_w1_bins", "arrival_energy_rmse"]
    metric_cols = [c for c in metric_cols if c in quality_df.columns]
    qh = quality_df[["method_show"] + metric_cols].copy()
    qh = qh.drop_duplicates("method_show")
    qh = qh.set_index("method_show")

    norm = pd.DataFrame(index=qh.index)
    for c in metric_cols:
        x = qh[c].copy()
        worst = x.max(skipna=True)
        x = x.fillna(worst)
        lo, hi = float(x.min()), float(x.max())
        norm[c] = 0.0 if abs(hi - lo) < 1e-12 else (x - lo) / (hi - lo)

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(norm.loc[[m for m in ord_save if m in norm.index]], cmap="YlGnBu", vmin=0, vmax=1, annot=True, fmt=".2f", cbar_kws={"label": "Normalized error (0=best, 1=worst)"}, ax=ax)
    ax.set_title("Forecast Error Heatmap (Method-level, Not EV/Charger-specific)")
    ax.set_xlabel("Metric")
    ax.set_ylabel("Method")
    fig.tight_layout()
    fig.savefig(os.path.join(fig_0701_dir, "fig08_forecast_error_heatmap.png"), dpi=320, bbox_inches="tight")
    plt.close(fig)
    add_manifest("fig08_forecast_error_heatmap.png", "How forecast errors compare across methods", "Heatmap values are normalized errors; 1 means worst among methods")

# fig09: skill map
if quality_df is not None and {"arrival_time_w1_bins", "arrival_energy_rmse"}.issubset(quality_df.columns):
    merged = quality_df.merge(res_d[res_d["case"] == "EV"][["method", save_col]], on="method", how="left")
    fig, ax = plt.subplots(figsize=(10, 7))
    size = 80 + 8 * merged[save_col].fillna(0).clip(lower=-30, upper=60).values
    ax.scatter(merged["arrival_time_w1_bins"], merged["arrival_energy_rmse"], s=size, color=IEEE["sky"], alpha=0.85)
    for _, r in merged.iterrows():
        ax.text(r["arrival_time_w1_bins"] + 0.04, r["arrival_energy_rmse"] + 0.04, r["method_show"], fontsize=8)
    ax.set_title("Forecast Skill Map (EV-side saving as bubble size)")
    ax.set_xlabel("Arrival-time distribution error (W1 bins)")
    ax.set_ylabel("Arrival-energy RMSE (kWh/bin)")
    fig.tight_layout()
    fig.savefig(os.path.join(fig_0701_dir, "fig09_skill_map_time_vs_energy.png"), dpi=320, bbox_inches="tight")
    plt.close(fig)
    add_manifest("fig09_skill_map_time_vs_energy.png", "Which methods jointly improve timing and energy prediction", "Bubble size uses EV-side strict-fair saving")

# fig10/11/12/14: probabilistic series
if prob_df is not None:
    x = prob_df["interval"].values
    y = prob_df["actual_arrival_energy"].values

    # fig10 intervals vs actual
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(x, y, color=IEEE["black"], linewidth=2, label="Perfect/Actual")
    if {"cp_q10", "cp_q90", "cp_p50"}.issubset(prob_df.columns):
        ax.fill_between(x, prob_df["cp_q10"], prob_df["cp_q90"], color=IEEE["blue"], alpha=0.18, label="iT-CP band")
        ax.plot(x, prob_df["cp_p50"], color=IEEE["blue"], linewidth=1.8, label="iT-CP p50")
    if {"tft_q10", "tft_q90", "tft_p50"}.issubset(prob_df.columns):
        ax.fill_between(x, prob_df["tft_q10"], prob_df["tft_q90"], color=IEEE["orange"], alpha=0.12, label="TFT band")
        ax.plot(x, prob_df["tft_p50"], color=IEEE["orange"], linewidth=1.6, label="TFT p50")
    if {"deepar_q10", "deepar_q90", "deepar_p50"}.issubset(prob_df.columns):
        ax.fill_between(x, prob_df["deepar_q10"], prob_df["deepar_q90"], color=IEEE["green"], alpha=0.12, label="DeepAR band")
        ax.plot(x, prob_df["deepar_p50"], color=IEEE["green"], linewidth=1.6, label="DeepAR p50")
    ax.set_title("Probabilistic Forecast Intervals vs Perfect Energy Time Series")
    ax.set_xlabel("15-min interval")
    ax.set_ylabel("Arrival energy (kWh/bin)")
    ax.legend(ncol=3, frameon=False)
    fig.tight_layout()
    fig.savefig(os.path.join(fig_0701_dir, "fig10_prob_interval_vs_actual.png"), dpi=320, bbox_inches="tight")
    plt.close(fig)
    add_manifest("fig10_prob_interval_vs_actual.png", "How probabilistic intervals cover the perfect energy series", "Bands are q10-q90")

    # fig11 interval width
    fig, ax = plt.subplots(figsize=(14, 4.8))
    if {"cp_q10", "cp_q90"}.issubset(prob_df.columns):
        ax.plot(x, prob_df["cp_q90"] - prob_df["cp_q10"], color=IEEE["blue"], label="iT-CP width")
    if {"tft_q10", "tft_q90"}.issubset(prob_df.columns):
        ax.plot(x, prob_df["tft_q90"] - prob_df["tft_q10"], color=IEEE["orange"], label="TFT width")
    if {"deepar_q10", "deepar_q90"}.issubset(prob_df.columns):
        ax.plot(x, prob_df["deepar_q90"] - prob_df["deepar_q10"], color=IEEE["green"], label="DeepAR width")
    ax.set_title("Probabilistic Interval Width (Uncertainty Sharpness)")
    ax.set_xlabel("15-min interval")
    ax.set_ylabel("Width (kWh/bin)")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(os.path.join(fig_0701_dir, "fig11_prob_coverage_width.png"), dpi=320, bbox_inches="tight")
    plt.close(fig)
    add_manifest("fig11_prob_coverage_width.png", "How sharp/wide each probabilistic method is", "Smaller width indicates sharper intervals")

    # fig12 residual series
    fig, ax = plt.subplots(figsize=(14, 4.8))
    if "cp_p50" in prob_df.columns:
        ax.plot(x, prob_df["cp_p50"] - y, color=IEEE["blue"], linewidth=1.6, label="iT-CP residual")
    if "tft_p50" in prob_df.columns:
        ax.plot(x, prob_df["tft_p50"] - y, color=IEEE["orange"], linewidth=1.4, label="TFT residual")
    if "deepar_p50" in prob_df.columns:
        ax.plot(x, prob_df["deepar_p50"] - y, color=IEEE["green"], linewidth=1.4, label="DeepAR residual")
    ax.axhline(0, color=IEEE["black"], linewidth=1)
    ax.set_title("Forecast Residual Time Series (Predicted - Perfect)")
    ax.set_xlabel("15-min interval")
    ax.set_ylabel("Residual (kWh/bin)")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(os.path.join(fig_0701_dir, "fig12_prob_residual_timeseries.png"), dpi=320, bbox_inches="tight")
    plt.close(fig)
    add_manifest("fig12_prob_residual_timeseries.png", "When each probabilistic method over/under predicts", "Residual = predicted minus perfect")

    # fig14 direct energy timeseries comparison
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(x, y, color=IEEE["black"], linewidth=2.2, label="Perfect/Actual")
    if "cp_p50" in prob_df.columns:
        ax.plot(x, prob_df["cp_p50"], color=IEEE["blue"], linewidth=1.9, label="iT-CP50")
    if "tft_p50" in prob_df.columns:
        ax.plot(x, prob_df["tft_p50"], color=IEEE["orange"], linewidth=1.8, label="TFT50")
    if "deepar_p50" in prob_df.columns:
        ax.plot(x, prob_df["deepar_p50"], color=IEEE["green"], linewidth=1.8, label="DeepAR50")
    ax.set_title("Forecast Energy Time Series vs Perfect")
    ax.set_xlabel("15-min interval")
    ax.set_ylabel("Arrival energy (kWh/bin)")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(os.path.join(fig_0701_dir, "fig14_forecast_energy_timeseries_vs_perfect.png"), dpi=320, bbox_inches="tight")
    plt.close(fig)
    add_manifest("fig14_forecast_energy_timeseries_vs_perfect.png", "How forecasted energy trajectories differ from perfect", "Direct time-series comparison")

# fig13: energy-gap before vs strict after
if "energy_gap_pct" in res_d.columns:
    compare = res_d[["method_show", "case", "energy_gap_pct"]].copy()
    compare["energy_gap_pct_strict"] = 0.0
    fig, axes = plt.subplots(1, 2, figsize=(15, 7), sharey=True)

    for ax_i, case in enumerate(["EV", "Charger"]):
        sub = compare[compare["case"] == case].set_index("method_show").reindex(ord_save).reset_index()
        y = np.arange(len(sub))
        axes[ax_i].barh(y - 0.2, sub["energy_gap_pct"], height=0.38, color=IEEE["orange"], label="Original")
        axes[ax_i].barh(y + 0.2, sub["energy_gap_pct_strict"], height=0.38, color=IEEE["blue"], label="Strict equal-energy")
        axes[ax_i].set_yticks(y)
        axes[ax_i].set_yticklabels(sub["method_show"])
        axes[ax_i].set_title(f"{case} energy gap")
        axes[ax_i].set_xlabel("Energy gap (%)")
        axes[ax_i].axvline(0, color=IEEE["black"], linewidth=1)
    axes[0].legend(frameon=False)
    fig.suptitle("Energy Gap Before vs Strict Equal-Energy Comparison")
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(os.path.join(fig_0701_dir, "fig13_energy_gap_before_after.png"), dpi=320, bbox_inches="tight")
    plt.close(fig)
    add_manifest("fig13_energy_gap_before_after.png", "Why strict equal-energy comparison is required", "Strict setting forces 0% energy gap")

# keep old names for backward compatibility
legacy_map = {
    "fig01_load_curves_all_methods.png": "0701_fig01_load_curves_all_methods.png",
    "fig02_cost_barh.png": "0701_fig02_cost_barh.png",
    "fig03_saving_grouped.png": "0701_fig03_saving_grouped.png",
    "fig04_dumbbell_ev_vs_charger_cost.png": "0701_fig04_dumbbell_ev_vs_charger_cost.png",
    "fig05_peak_cost_bubble.png": "0701_fig05_peak_cost_bubble.png",
    "fig06_error_vs_saving.png": "0701_fig06_error_vs_saving.png",
    "fig07_rank_card.png": "0701_fig07_rank_card.png",
}
for new_name, old_name in legacy_map.items():
    new_path = os.path.join(fig_0701_dir, new_name)
    old_path = os.path.join(fig_0701_dir, old_name)
    if os.path.exists(new_path):
        shutil.copy2(new_path, old_path)

manifest_seq = pd.DataFrame(manifest_rows)
manifest_seq.to_csv(os.path.join(CONFIG["table_dir"], "figure_manifest_0701.csv"), index=False)
res_d.to_csv(os.path.join(CONFIG["table_dir"], "result_0701_only.csv"), index=False)

print("Part 6 complete")
print("Saved sequential figures in:", fig_0701_dir)
print("Primary manifest:", os.path.join(CONFIG["table_dir"], "figure_manifest_0701.csv"))


Part 6 complete
Saved sequential figures in: /Users/admin/Desktop/EV_program/2024Summer_EVResearch/2026_Gu_EV_forecast/figures/0701
Primary manifest: /Users/admin/Desktop/EV_program/2024Summer_EVResearch/2026_Gu_EV_forecast/tables/figure_manifest_0701.csv


## Part 7: Final Sanity Check and One-Day Conclusions (Strict Equal-Energy)

This final block verifies strict fairness constraints: all methods are compared under equal total dispatched energy.


In [12]:
# Final one-day sanity check and concise conclusions (strict equal-energy)

print("=" * 90)
print("FINAL SUMMARY (SINGLE-DAY 0701, STRICT ENERGY-FAIR)")
print("=" * 90)

focus_day = CONFIG["test_days"][0]
res = pd.read_csv(os.path.join(CONFIG["table_dir"], "result_0701_only.csv"))
res["day"] = pd.to_datetime(res["day"]).dt.date
res = res[res["day"] == focus_day].copy()

if len(res) == 0:
    raise RuntimeError("No 0701 records available for final summary")

cost_col = "mpc_cost_strict" if "mpc_cost_strict" in res.columns else "mpc_cost"
save_col = "cost_saving_pct_strict" if "cost_saving_pct_strict" in res.columns else ("effective_saving_pct" if "effective_saving_pct" in res.columns else "cost_saving_pct")
energy_col = "energy_kwh_strict" if "energy_kwh_strict" in res.columns else "energy_kwh"
gap_col = "energy_gap_pct_strict" if "energy_gap_pct_strict" in res.columns else "energy_gap_pct"
pass_col = "sanity_energy_pass_strict" if "sanity_energy_pass_strict" in res.columns else "sanity_energy_pass"

summary = (
    res.groupby(["method", "case"], as_index=False)
    .agg(
        mean_cost=(cost_col, "mean"),
        mean_saving_pct=(save_col, "mean"),
        mean_peak=("peak_kw", "mean"),
        mean_energy=(energy_col, "mean"),
        mean_optimal_ratio=("status_ratio_optimal", "mean"),
    )
)
summary["rank_in_case"] = summary.groupby("case")["mean_cost"].rank(method="dense", ascending=True).astype(int)
summary = summary.sort_values(["case", "rank_in_case", "method"]).reset_index(drop=True)

best_ev = summary[summary["case"] == "EV"].sort_values("mean_cost").iloc[0]
best_ch = summary[summary["case"] == "Charger"].sort_values("mean_cost").iloc[0]

print("Data coverage")
print("- benchmark day:", focus_day)
print("- methods:", sorted(res["method"].unique()))
print("- cases:", sorted(res["case"].unique()))
print("- total records:", len(res))
print("- mean status ratio:", round(res["status_ratio_optimal"].mean(), 4))

print("\nBest method by case (strict-fair cost)")
print(f"- EV: {best_ev['method']} | cost={best_ev['mean_cost']:.2f} | saving={best_ev['mean_saving_pct']:.2f}%")
print(f"- Charger: {best_ch['method']} | cost={best_ch['mean_cost']:.2f} | saving={best_ch['mean_saving_pct']:.2f}%")

if gap_col in res.columns:
    print("\nEnergy-gap sanity")
    for case in ["EV", "Charger"]:
        sub = res[res["case"] == case][gap_col]
        print(f"- {case}: min={sub.min():.6f}%, mean={sub.mean():.6f}%, max={sub.max():.6f}%")

if pass_col in res.columns:
    print("\nEnergy-pass ratio")
    for case in ["EV", "Charger"]:
        sub = res[res["case"] == case][pass_col].astype(bool)
        print(f"- {case}: {sub.mean() * 100:.1f}%")

summary_out = os.path.join(CONFIG["table_dir"], "summary_by_method_case.csv")
pub_out = os.path.join(CONFIG["table_dir"], "publication_main_table.csv")
summary.to_csv(summary_out, index=False)
summary.to_csv(pub_out, index=False)

print("\nSaved summary tables")
print("-", summary_out)
print("-", pub_out)
print("-", os.path.join(CONFIG["table_dir"], "result_0701_strict_equal_energy.csv"))
print("-", os.path.join(CONFIG["table_dir"], "sanity_check_0701_strict.csv"))
print("-", os.path.join(CONFIG["table_dir"], "figure_manifest_0701.csv"))
print("=" * 90)


FINAL SUMMARY (SINGLE-DAY 0701, STRICT ENERGY-FAIR)
Data coverage
- benchmark day: 2023-07-01
- methods: ['DeepAR-Gaussian', 'GMM', 'ITCN-ChargerAware', 'LSTM', 'Noforecast', 'Perfect', 'Persistence', 'Statistic', 'TCN', 'TFT-Quantile', 'Transformer', 'iT-CP90', 'iT-CP95', 'iTransformer']
- cases: ['Charger', 'EV']
- total records: 28
- mean status ratio: 0.9457

Best method by case (strict-fair cost)
- EV: Perfect | cost=2259.21 | saving=46.82%
- Charger: Perfect | cost=2283.74 | saving=46.25%

Energy-gap sanity
- EV: min=0.000000%, mean=0.000000%, max=0.000000%
- Charger: min=0.000000%, mean=0.000000%, max=0.000000%

Energy-pass ratio
- EV: 100.0%
- Charger: 100.0%

Saved summary tables
- /Users/admin/Desktop/EV_program/2024Summer_EVResearch/2026_Gu_EV_forecast/tables/summary_by_method_case.csv
- /Users/admin/Desktop/EV_program/2024Summer_EVResearch/2026_Gu_EV_forecast/tables/publication_main_table.csv
- /Users/admin/Desktop/EV_program/2024Summer_EVResearch/2026_Gu_EV_forecast/tables

In [15]:
# Part 8: Consolidated forecast-curve figure (all methods vs Perfect)

fig_0701_dir = os.path.join(CONFIG["fig_dir"], "0701")
os.makedirs(fig_0701_dir, exist_ok=True)
focus_day = CONFIG["test_days"][0]

name_map = {
    "Noforecast": "NoForecast",
    "iT-CP90": "iT-CP90",
    "iT-CP95": "iT-CP95",
    "TFT-Quantile": "TFT",
    "DeepAR-Gaussian": "DeepAR",
    "ITCN-ChargerAware": "ITCN-ChargerAware",
}

actual_day = sessions_all[sessions_all["day"] == focus_day].copy()
actual_std = standardize_session_df(actual_day[["session_id", "charger_id", "AT_idx", "DT_idx", "ED"]], focus_day)
hist = choose_training_pool(sessions_all, focus_day)

_, perfect_curve, _ = build_daily_interval_profile(actual_std)

method_order = [m for m in CONFIG["methods"] if m in set(results_df["method"])]
non_perfect = [m for m in method_order if m != "Perfect"]

profile_bank = {}
for m in non_perfect:
    forecast_df = forecast_sessions(m, focus_day, actual_std, hist, stats_bank, model_bank)
    _, arr_energy, _ = build_daily_interval_profile(forecast_df)
    profile_bank[m] = arr_energy

split_idx = (len(non_perfect) + 1) // 2
groups = [non_perfect[:split_idx], non_perfect[split_idx:]]

fig, axes = plt.subplots(2, 1, figsize=(15, 10), sharex=True, sharey=True)
x = np.arange(1, CONFIG["N"] + 1)

for panel_i, (ax, group) in enumerate(zip(axes, groups), start=1):
    # Always plot Perfect curve as reference.
    ax.plot(x, perfect_curve, color=IEEE["black"], linewidth=2.4, label="Perfect/Actual")

    zero_methods = []
    for m in group:
        curve = np.asarray(profile_bank[m], dtype=float)
        if float(np.max(np.abs(curve))) <= 1e-6:
            zero_methods.append(name_map.get(m, m))
        else:
            ax.plot(x, curve, linewidth=1.4, alpha=0.9, label=name_map.get(m, m))

    ax.set_ylabel("Arrival energy (kWh/bin)")
    ax.set_title(f"Batch {panel_i}: forecast methods vs Perfect (non-zero curves only)")
    ax.legend(ncol=4, frameon=False, fontsize=8, loc="upper right")

    if len(zero_methods) > 0:
        zero_text = "All-zero methods: " + ", ".join(zero_methods)
        ax.text(
            0.01,
            0.97,
            zero_text,
            transform=ax.transAxes,
            va="top",
            ha="left",
            fontsize=8,
            bbox={"boxstyle": "round,pad=0.25", "facecolor": "white", "alpha": 0.75, "edgecolor": "#666666"},
        )

axes[-1].set_xlabel("15-min interval")
fig.suptitle(f"All Forecast Methods vs Perfect on {focus_day} (Two-Panel)", fontsize=16, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig10_name = "fig10_all_methods_vs_perfect_two_panel.png"
fig.savefig(os.path.join(fig_0701_dir, fig10_name), dpi=320, bbox_inches="tight")
plt.close(fig)

# Keep only latest figure set: fig01~fig10
keep_fig = {
    "fig01_load_curves_all_methods.png",
    "fig02_cost_barh.png",
    "fig03_saving_grouped.png",
    "fig04_dumbbell_ev_vs_charger_cost.png",
    "fig05_peak_cost_bubble.png",
    "fig06_error_vs_saving.png",
    "fig07_rank_card.png",
    "fig08_forecast_error_heatmap.png",
    "fig09_skill_map_time_vs_energy.png",
    fig10_name,
}

removed_fig = []
for fname in os.listdir(fig_0701_dir):
    fpath = os.path.join(fig_0701_dir, fname)
    if os.path.isfile(fpath) and fname not in keep_fig:
        os.remove(fpath)
        removed_fig.append(fname)

# Update manifest to align with the new figure set
manifest_path = os.path.join(CONFIG["table_dir"], "figure_manifest_0701.csv")
if os.path.exists(manifest_path):
    manifest = pd.read_csv(manifest_path)
else:
    manifest = pd.DataFrame(columns=["file", "core_question", "note"])

manifest = manifest[manifest["file"].isin(keep_fig)].copy()
manifest = manifest[manifest["file"] != fig10_name]
manifest = pd.concat(
    [
        manifest,
        pd.DataFrame(
            [{
                "file": fig10_name,
                "core_question": "How all forecast methods track the perfect energy time series",
                "note": "Non-zero methods are plotted; all-zero methods are annotated as text per panel",
            }]
        ),
    ],
    ignore_index=True,
)

manifest["_ord"] = manifest["file"].str.extract(r"fig(\d+)").astype(float)
manifest = manifest.sort_values("_ord").drop(columns=["_ord"]).reset_index(drop=True)
manifest.to_csv(manifest_path, index=False)

print("Part 8 complete")
print("Saved:", os.path.join(fig_0701_dir, fig10_name))
print("Removed old figures:", len(removed_fig))
print("Updated manifest:", manifest_path)

Part 8 complete
Saved: /Users/admin/Desktop/EV_program/2024Summer_EVResearch/2026_Gu_EV_forecast/figures/0701/fig10_all_methods_vs_perfect_two_panel.png
Removed old figures: 0
Updated manifest: /Users/admin/Desktop/EV_program/2024Summer_EVResearch/2026_Gu_EV_forecast/tables/figure_manifest_0701.csv
